# 01c — MCI Time-to-Dementia **Survival Cohort Freeze** (Build 1, `v1`)

Creates the **authoritative, analysis-ready frozen cohort** for the study *"does baseline plasma
p-tau217 improve prediction of MCI→dementia progression beyond age and APOE4 count?"*

This notebook **re-derives the cohort from raw source data** and freezes three nested participant-level
tables, a participant-level exclusion log, a cohort-flow table, and a provenance manifest. The
downstream modeling owner (Person 2) can then work entirely from the frozen primary table with
**no raw-data joins**.

## What this notebook is NOT
No Cox model, no feature selection, no transformation/standardisation, no imputation, no risk groups,
no QC sampling (Build 2), no p-tau181 integration. It **builds and validates data only**.

## Provenance of the scientific rules
Every scientific rule is inherited **unchanged** from `01b_mci_survival_cohort_audit.ipynb`. The
canonical functions `identify_broad_anchor` and `derive_survival` are reused **verbatim**;
`align_scores` carries a single **additive-only** change (it now also returns the matched score date
for provenance — the `merge_asof` selection, tolerance and direction are byte-identical).

A regression cell at the end asserts this freeze reproduces the `01b` master artifact **exactly** on
every shared scientific column. `01b` and its outputs are **read only, never overwritten**.

## Locked rules (do not change without team approval)
| Rule | Value |
|---|---|
| Diagnosis coding | `1=CN, 2=MCI, 3=Dementia`; blank → `unknown/other` (never event, never censor) |
| Same-day diagnosis conflict | collapse per `(RID, DATE)` keeping **highest severity** |
| Missing sentinels | `-4`, `-5` → `NaN` |
| Non-positive assay value | → `NaN` (plasma concentrations must be > 0) |
| Diagnosis↔plasma alignment | **nearest** MCI diagnosis within **±90 days** |
| Broad anchor | **earliest** eligible plasma draw with **≥1 usable core assay**; does **not** require p-tau217, APOE4, age, cognition, or follow-up |
| Anchor tie-break | earliest plasma `DATE`; plasma pre-deduplicated to one row per `(RID, DATE)` (`keep="first"`); stable original row order |
| Time origin | plasma anchor date |
| Event | **first** dementia diagnosis **strictly after** the anchor |
| Censoring | **last** post-anchor non-dementia (CN/MCI) diagnosis date |
| No usable post-anchor follow-up | **excluded** — never labelled "stable" |
| Primary complete case | survival-eligible **+** `entry_age` **+** `APOE4_COUNT` **+** `ptau217` |
| Age | `entry_age` is authoritative (**undated** — documented caveat); age-at-anchor is *not* derived |
| Imputation | none |

## 0. Setup, paths, locked configuration

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


def find_project_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "Data" / "raw").is_dir():
            return cand
    raise FileNotFoundError("Could not locate Data/raw above %s" % start)


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "Data" / "raw"
OUT_DIR = PROJECT_ROOT / "outputs" / "01c_mci_survival_cohort_freeze"
PRIOR_DIR = PROJECT_ROOT / "outputs" / "01b_mci_survival_cohort_audit"   # READ-ONLY
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- LOCKED configuration: identical to 01b. Do not change. ----------------
MISSING_SENTINELS = [-4, -5]
NONPOSITIVE_IS_MISSING = True
ALIGN_TOL_DAYS = 90
DAYS_PER_YEAR = 365.25
DAYS_PER_MONTH = 30.4375          # = 365.25 / 12 — the ONE documented day->month conversion
CORE_ASSAYS = ["ptau217", "abeta42", "abeta40", "nfl", "gfap"]
RANDOM_SEED = 20260711            # recorded for reproducibility; NO sampling occurs in Build 1
VERSION = "v1"

# Reconciliation targets from the 01b audit. Used for REPORTING ONLY -- never to force a result.
TARGETS = {"anchor": 535, "followup": 410, "followup_events": 86, "primary": 401, "primary_events": 85}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR     :", OUT_DIR)
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

PROJECT_ROOT: /Users/zoeyd/Desktop/Predict_AD_from_Biomarkers
OUT_DIR     : /Users/zoeyd/Desktop/Predict_AD_from_Biomarkers/outputs/01c_mci_survival_cohort_freeze
python 3.14.0 | pandas 3.0.1 | numpy 2.4.3


### Assertion harness

Every check is registered and **fails loudly** with participant-level diagnostics. A failure raises,
which aborts the notebook — the cohort is never declared frozen on a failed assertion.

In [2]:
ASSERTIONS: list[dict] = []


def check(name: str, ok, detail: str = "") -> None:
    """Register an assertion. Raise loudly with diagnostics if it fails."""
    ok = bool(ok)
    ASSERTIONS.append({"assertion": name, "passed": ok, "detail": detail})
    if not ok:
        raise AssertionError(f"ASSERTION FAILED: {name}\n  diagnostics: {detail}")
    print(f"  PASS  {name}" + (f"  [{detail}]" if detail else ""))


def rid_list(rids, limit: int = 20) -> str:
    """Compact, interpretable participant-level diagnostic."""
    rids = sorted(int(r) for r in rids)
    head = ", ".join(str(r) for r in rids[:limit])
    return f"n={len(rids)} RIDs=[{head}{', ...' if len(rids) > limit else ''}]"


def save_tsv(df: pd.DataFrame, name: str) -> Path:
    path = OUT_DIR / name
    df.to_csv(path, sep="\t", index=False)
    print(f"  saved -> {path.relative_to(PROJECT_ROOT)}   ({df.shape[0]} x {df.shape[1]})")
    return path


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

## 1. Load raw sources + generate deterministic source-row identifiers

**Source-row provenance rule.** Only `DXSUM` and `APOERES` ship a native ADNI row id (`ID`); the
UPENN plasma panel and `My_Table` have **none**. So for *every* source table we generate:

> `<prefix>_src_row` = the **0-based positional index of the record in the raw CSV as delivered**
> (row `0` = first data row after the header), assigned **immediately after `pd.read_csv` and before
> any filtering, sorting, de-duplication or merging**.

This is stable and reproducible for a fixed source file. Where a native `ID` exists it is carried
**as well** (`dx_src_id`, `apoe_src_id`) so rows can be traced either way.

In [3]:
def load_raw(fname: str, prefix: str, **kw) -> pd.DataFrame:
    """Read a raw CSV and stamp a deterministic 0-based source-row index BEFORE any processing."""
    df = pd.read_csv(RAW_DIR / fname, **kw)
    df[f"{prefix}_src_row"] = np.arange(len(df), dtype="int64")
    return df


SRC_FILES = {
    "dxsum":  "All_Subjects_DXSUM_05Mar2026.csv",
    "plasma": "All_Subjects_UPENN_PLASMA_FUJIREBIO_QUANTERIX_05Mar2026.csv",
    "apoe":   "All_Subjects_APOERES_05Mar2026.csv",
    "age":    "All_Subjects_My_Table_05Mar2026.csv",
    "cdr":    "All_Subjects_CDR_05Mar2026.csv",
    "mmse":   "All_Subjects_MMSE_05Mar2026.csv",
    "moca":   "All_Subjects_MOCA_05Mar2026.csv",
    "faq":    "All_Subjects_FAQ_05Mar2026.csv",
}

dxsum = load_raw(SRC_FILES["dxsum"], "dx",
                 usecols=["ID", "PTID", "RID", "VISCODE2", "EXAMDATE", "DIAGNOSIS"]
                 ).rename(columns={"ID": "dx_src_id"})
plasma = load_raw(SRC_FILES["plasma"], "plasma",
                  usecols=["PTID", "RID", "VISCODE2", "EXAMDATE",
                           "pT217_F", "AB42_F", "AB40_F", "AB42_AB40_F", "NfL_Q", "GFAP_Q"])
apoe = load_raw(SRC_FILES["apoe"], "apoe",
                usecols=["ID", "RID", "GENOTYPE"]).rename(columns={"ID": "apoe_src_id"})
age = load_raw(SRC_FILES["age"], "age")                     # subject_id (==PTID), entry_age
cdr = load_raw(SRC_FILES["cdr"], "cdr", usecols=["RID", "VISDATE", "CDVERSION", "CDRSB"])
mmse = load_raw(SRC_FILES["mmse"], "mmse", usecols=["RID", "VISDATE", "MMSCORE"])
moca = load_raw(SRC_FILES["moca"], "moca", usecols=["RID", "VISDATE", "MOCA"])
faq = load_raw(SRC_FILES["faq"], "faq", usecols=["RID", "VISDATE", "FAQTOTAL"])

# The UPENN plasma panel carries NO native row id -> the generated index is the only anchor row id.
print("plasma has native 'ID' column:", "ID" in pd.read_csv(RAW_DIR / SRC_FILES["plasma"], nrows=0).columns)
print("Loaded:", {k: len(v) for k, v in
                  dict(dxsum=dxsum, plasma=plasma, apoe=apoe, age=age,
                       cdr=cdr, mmse=mmse, moca=moca, faq=faq).items()})

# p-tau181 is deliberately NOT integrated in v1 (Build 1 decision, see report).
PTAU181_STATUS = ("not integrated in v1: p-tau181 exists only in "
                  "All_Subjects_FNIHBC_BLOOD_BIOMARKER_TRAJECTORIES_05Mar2026.csv "
                  "(long format, 2 platforms: QX_plasma_ptau181 / Roche_plasma_ptau181); that "
                  "sub-study overlaps only 40/535 anchor participants (33/410 survival-eligible). "
                  "Its absence does not affect anchor, survival or primary eligibility.")
print("\np-tau181:", PTAU181_STATUS)

plasma has native 'ID' column: False
Loaded: {'dxsum': 16088, 'plasma': 2178, 'apoe': 3253, 'age': 5031, 'cdr': 14734, 'mmse': 14709, 'moca': 9072, 'faq': 13385}

p-tau181: not integrated in v1: p-tau181 exists only in All_Subjects_FNIHBC_BLOOD_BIOMARKER_TRAJECTORIES_05Mar2026.csv (long format, 2 platforms: QX_plasma_ptau181 / Roche_plasma_ptau181); that sub-study overlaps only 40/535 anchor participants (33/410 survival-eligible). Its absence does not affect anchor, survival or primary eligibility.


## 2. Diagnosis harmonization  *(verbatim from `01b`)*

`1=CN, 2=MCI, 3=Dementia`; blank → `unknown/other` (never used to define an event or a censor).
Rows without a date or without a diagnosis are dropped. Same-day records for one participant are
collapsed keeping the **highest severity** (sort by `dx_code` ascending, keep last).

In [4]:
def clean_sentinels(s: pd.Series) -> pd.Series:
    """01b, verbatim. ADNI missing codes -4/-5 -> NaN."""
    v = pd.to_numeric(s, errors="coerce")
    return v.mask(v.isin(MISSING_SENTINELS))


def harmonize_diagnosis(code: float) -> str:
    """01b, verbatim."""
    return {1.0: "CN", 2.0: "MCI", 3.0: "Dementia"}.get(code, "unknown/other")


dx = dxsum.copy()
dx["DATE"] = pd.to_datetime(dx["EXAMDATE"], errors="coerce")
dx["dx_code"] = pd.to_numeric(dx["DIAGNOSIS"], errors="coerce")
dx["dx_harmonized"] = dx["dx_code"].map(harmonize_diagnosis)

n_no_date = int(dx["DATE"].isna().sum())
n_no_dx = int(dx["dx_code"].isna().sum())
dxh = dx.dropna(subset=["DATE", "dx_code"]).sort_values(["RID", "DATE", "dx_code"])
n_conflict_sameday = int(dxh.duplicated(["RID", "DATE"], keep=False).sum())
dxh = dxh.drop_duplicates(["RID", "DATE"], keep="last")     # tie-break: highest severity same-day

print(f"Dropped {n_no_date} rows w/o date, {n_no_dx} w/o diagnosis. "
      f"Same-day records collapsed (highest severity): {n_conflict_sameday} rows.")
print(f"Clean dx history: {len(dxh)} visits, {dxh['RID'].nunique()} participants.")

# dxh is unique on (RID, DATE) -> it is a valid provenance key for anchor/event/censor rows.
check("dxh unique on (RID, DATE)  [source-row join key is well-defined]",
      not dxh.duplicated(["RID", "DATE"]).any())
DX_PROV = dxh[["RID", "DATE", "dx_src_row", "dx_src_id"]].copy()

Dropped 115 rows w/o date, 45 w/o diagnosis. Same-day records collapsed (highest severity): 76 rows.
Clean dx history: 15935 visits, 3763 participants.
  PASS  dxh unique on (RID, DATE)  [source-row join key is well-defined]


## 3. Plasma panel cleaning  *(verbatim from `01b`)*

Sentinels (−4, −5) → `NaN`; non-positive concentrations → `NaN`. An assay is **usable** at a draw if
it is present, non-sentinel and > 0. The broad anchor needs **≥ 1** usable core assay among
p-tau217 / Aβ42 / Aβ40 / NfL / GFAP. Raw values are preserved — nothing is transformed.

In [5]:
plasma_c = plasma.copy()
plasma_c["DATE"] = pd.to_datetime(plasma_c["EXAMDATE"], errors="coerce")

SRC = {"ptau217": "pT217_F", "abeta42": "AB42_F", "abeta40": "AB40_F",
       "nfl": "NfL_Q", "gfap": "GFAP_Q", "ratio_ab42_ab40_vendor": "AB42_AB40_F"}

suspicious = {}
for name, col in SRC.items():
    v = clean_sentinels(plasma_c[col])
    if NONPOSITIVE_IS_MISSING:
        bad = v.notna() & (v <= 0)
        suspicious[name] = int(bad.sum())
        v = v.mask(v <= 0)
    plasma_c[name] = v

plasma_c["ratio_ab42_ab40"] = plasma_c["abeta42"] / plasma_c["abeta40"]
plasma_c["n_usable_core_assays"] = plasma_c[CORE_ASSAYS].notna().sum(axis=1)

n_dupe = int(plasma_c.duplicated(["RID", "DATE"], keep=False).sum())
plasma_c = (plasma_c.dropna(subset=["DATE"])
                    .sort_values(["RID", "DATE"])
                    .drop_duplicates(["RID", "DATE"], keep="first"))

print("Suspicious non-positive values -> NaN:", suspicious)
print(f"Plasma duplicate (RID,DATE) collapsed: {n_dupe}. Clean plasma rows: {len(plasma_c)}.")
print("Draws with >=1 usable core assay:",
      int((plasma_c["n_usable_core_assays"] >= 1).sum()), "of", len(plasma_c))

check("plasma unique on (RID, DATE) after de-duplication  [anchor tie-break is fully determined]",
      not plasma_c.duplicated(["RID", "DATE"]).any())

Suspicious non-positive values -> NaN: {'ptau217': 0, 'abeta42': 0, 'abeta40': 0, 'nfl': 0, 'gfap': 0, 'ratio_ab42_ab40_vendor': 0}
Plasma duplicate (RID,DATE) collapsed: 0. Clean plasma rows: 2178.
Draws with >=1 usable core assay: 2178 of 2178
  PASS  plasma unique on (RID, DATE) after de-duplication  [anchor tie-break is fully determined]


## 4. APOE and age  *(verbatim from `01b`, + source-row provenance)*

- `APOE4_COUNT` = number of ε4 alleles parsed from `GENOTYPE` (time-invariant).
- **Age (locked Build 1 decision):** `entry_age` from `My_Table` is the **authoritative** baseline age.
  It carries **no datestamp** in its source file, so **age at the plasma anchor is not derived**.
  For transparency we still carry the *flagged, non-authoritative* provenance fields
  `study_entry_date_proxy` (earliest dated DXSUM visit), `years_entry_to_anchor` and
  `age_at_anchor_approx`. **These must not be used as the model's age variable.**
  No participant is excluded for missing age *timing* — only the existing `entry_age` missingness rule applies.

In [6]:
apoe_c = apoe.copy()
apoe_c["APOE4_COUNT"] = apoe_c["GENOTYPE"].astype(str).str.count("4")
apoe_c = (apoe_c.dropna(subset=["GENOTYPE"])
                .drop_duplicates("RID")[["RID", "GENOTYPE", "APOE4_COUNT", "apoe_src_row", "apoe_src_id"]])

ptid_rid = dxsum[["PTID", "RID"]].drop_duplicates()
age_c = (age.rename(columns={"subject_id": "PTID"})
            .merge(ptid_rid, on="PTID", how="inner")
            .drop_duplicates("RID")[["RID", "entry_age", "age_src_row"]])

# study-entry proxy = earliest dated DXSUM visit per participant (NON-authoritative; provenance only)
entry_proxy = (dx.dropna(subset=["DATE"]).sort_values("DATE")
                 .groupby("RID")["DATE"].first().rename("study_entry_date_proxy").reset_index())

print(f"APOE4_COUNT available: {apoe_c['RID'].nunique()} | entry_age available: {age_c['RID'].nunique()}")
print(apoe_c["APOE4_COUNT"].value_counts().sort_index().to_string())

APOE4_COUNT available: 3253 | entry_age available: 3788
APOE4_COUNT
0    1779
1    1177
2     297


## 5. Broad MCI plasma anchor (Cohort A)  *(`identify_broad_anchor` — verbatim from `01b`)*

> The **earliest** plasma draw that (1) has a valid draw date, (2) comes from the primary plasma
> panel, (3) has **≥1 usable core assay**, and (4) aligns to a **nearest MCI** diagnosis within
> **±90 days**.

It does **not** require p-tau217, APOE4, age, cognition, a complete panel, or any follow-up.
A later, more-complete draw is **never** preferred — the anchor marks *when prediction begins*.

`plasma_src_row` flows through `merge_asof` automatically, so the selected anchor row is traceable
with **no change to the canonical function**. The matched-diagnosis source row is recovered
afterwards by an exact `(RID, dx_date)` key-join against `dxh` (unique on that key, asserted above).

In [7]:
def identify_broad_anchor(plasma_df, dxh_df, tol_days=ALIGN_TOL_DAYS):
    """01b `identify_broad_anchor`, VERBATIM. Any extra columns on `plasma_df` (e.g. plasma_src_row)
    are carried through merge_asof untouched."""
    left = plasma_df[plasma_df["n_usable_core_assays"] >= 1].dropna(subset=["DATE"]).sort_values("DATE")
    right = (dxh_df[["RID", "DATE", "dx_harmonized", "VISCODE2"]]
             .rename(columns={"DATE": "dx_date", "VISCODE2": "dx_viscode"})
             .dropna(subset=["dx_date"]).sort_values("dx_date"))
    m = pd.merge_asof(left, right, left_on="DATE", right_on="dx_date", by="RID",
                      tolerance=pd.Timedelta(days=tol_days), direction="nearest")
    m["align_offset_days"] = (m["DATE"] - m["dx_date"]).dt.days
    mci = m[m["dx_harmonized"] == "MCI"].copy()
    anchor = mci.sort_values(["RID", "DATE"]).drop_duplicates("RID", keep="first")
    return anchor


anchor = identify_broad_anchor(plasma_c, dxh)
anchor = anchor.rename(columns={"DATE": "anchor_date",
                                "VISCODE2": "anchor_viscode",
                                "dx_viscode": "anchor_dx_viscode",
                                "dx_date": "anchor_dx_date",
                                "plasma_src_row": "anchor_src_row"})

anchor["align_offset_days_abs"] = anchor["align_offset_days"].abs()
anchor["same_day_alignment"] = anchor["align_offset_days"].abs() == 0
anchor["anchor_match_type"] = np.where(anchor["same_day_alignment"], "same_day", "cross_visit")
anchor["anchor_tiebreak_rule"] = ("earliest eligible plasma DATE; plasma pre-deduplicated to one row "
                                  "per (RID,DATE) keep=first; stable original row order")
anchor["ptau217_assay"] = "Fujirebio pT217_F (UPENN_PLASMA_FUJIREBIO_QUANTERIX)"

# --- matched-MCI-diagnosis source row (exact key join; dxh unique on (RID,DATE)) ---
anchor = anchor.merge(
    DX_PROV.rename(columns={"DATE": "anchor_dx_date",
                            "dx_src_row": "anchor_dx_src_row",
                            "dx_src_id": "anchor_dx_src_id"}),
    on=["RID", "anchor_dx_date"], how="left", validate="m:1")

# --- age / APOE provenance merges (01b order) ---
anchor = anchor.merge(age_c, on="RID", how="left").merge(entry_proxy, on="RID", how="left")
anchor["years_entry_to_anchor"] = (anchor["anchor_date"] - anchor["study_entry_date_proxy"]).dt.days / DAYS_PER_YEAR
anchor["age_at_anchor_approx"] = anchor["entry_age"] + anchor["years_entry_to_anchor"]
anchor["age_is_entry_age_fallback"] = True
anchor["age_derivation_suspect"] = anchor["years_entry_to_anchor"] < 0
anchor["age_source_date"] = pd.NaT           # My_Table carries NO datestamp -- structurally absent
anchor["entry_age_has_datestamp"] = False

n_anchor = len(anchor)
print(f"Broad MCI plasma-anchor cohort (A): {n_anchor} participants")
print(f"  same-day alignment : {int(anchor['same_day_alignment'].sum())}"
      f"  | cross-visit: {int((~anchor['same_day_alignment']).sum())}")
print(f"  |offset| median/p90/max : {anchor['align_offset_days_abs'].median():.0f}"
      f"/{anchor['align_offset_days_abs'].quantile(.9):.0f}/{anchor['align_offset_days_abs'].max():.0f}")

Broad MCI plasma-anchor cohort (A): 535 participants
  same-day alignment : 154  | cross-visit: 381
  |offset| median/p90/max : 7/40/90


### 5b. Dementia on or before the anchor — **flag, do not exclude**

Spec §2.5 requires that a dementia diagnosis **on or before** the anchor is never counted as incident
progression, and that such cases are *"excluded **or flagged** for adjudication"*.

- **On the anchor date:** structurally impossible to be included — a same-day dementia would be the
  *nearest* diagnosis (offset 0 d) and the draw would not match as MCI. `derive_survival` also never
  treats it as a prospective event.
- **Before the anchor** (e.g. Dementia → reversion → MCI at anchor): not checked by `01b`. Under the
  **locked v1 rules such a participant is retained**, so here we **flag** rather than exclude
  (`qc_dementia_on_or_before_anchor`, `pre_anchor_dementia_n`) and surface the count for Build 2
  adjudication. Excluding them would change a locked rule and requires team approval.

In [8]:
_dem = dxh.loc[dxh["dx_harmonized"] == "Dementia", ["RID", "DATE"]]
_t = anchor[["RID", "anchor_date"]].merge(_dem, on="RID", how="left")
_t["_pre"] = _t["DATE"] < _t["anchor_date"]
_t["_same"] = _t["DATE"] == _t["anchor_date"]
_pre = (_t.groupby("RID")[["_pre", "_same"]].sum().reset_index()
          .rename(columns={"_pre": "pre_anchor_dementia_n", "_same": "sameday_dementia_dx_n"}))

anchor = anchor.merge(_pre, on="RID", how="left")
anchor[["pre_anchor_dementia_n", "sameday_dementia_dx_n"]] = (
    anchor[["pre_anchor_dementia_n", "sameday_dementia_dx_n"]].fillna(0).astype("int64"))
anchor["qc_dementia_on_or_before_anchor"] = (
    (anchor["pre_anchor_dementia_n"] + anchor["sameday_dementia_dx_n"]) > 0)

n_pre_dem = int(anchor["qc_dementia_on_or_before_anchor"].sum())
print(f"Dementia dx strictly BEFORE anchor : {int((anchor['pre_anchor_dementia_n'] > 0).sum())} participants")
print(f"Dementia dx ON the anchor date     : {int((anchor['sameday_dementia_dx_n'] > 0).sum())} participants")
print(f"-> FLAGGED for adjudication (not excluded): {n_pre_dem}")
if n_pre_dem:
    print("   RIDs:", rid_list(anchor.loc[anchor["qc_dementia_on_or_before_anchor"], "RID"]))

check("no dementia diagnosis ON the anchor date is treated as an incident event",
      int((anchor["sameday_dementia_dx_n"] > 0).sum()) == 0,
      "structurally guaranteed: a same-day dementia would be the nearest dx and would not match as MCI")

Dementia dx strictly BEFORE anchor : 8 participants
Dementia dx ON the anchor date     : 0 participants
-> FLAGGED for adjudication (not excluded): 8
   RIDs: n=8 RIDs=[467, 4115, 4506, 4706, 6976, 7070, 10251, 10322]
  PASS  no dementia diagnosis ON the anchor date is treated as an incident event  [structurally guaranteed: a same-day dementia would be the nearest dx and would not match as MCI]


## 6. Survival outcome and censoring (Cohort B)  *(`derive_survival` — verbatim from `01b`)*

- **Time origin:** the plasma anchor date.
- **Event:** the **first** dementia diagnosis **strictly after** the anchor (`DATE > anchor`) → `event_indicator = 1`.
- **Censoring:** no post-anchor dementia → censor at the **last** post-anchor non-dementia (CN/MCI) date → `event_indicator = 0`.
- **No usable post-anchor follow-up** → **excluded** from the survival cohort. Never called "stable".
- The `DATE > anchor` filter is the **leakage guard**: no future information can reach the anchor.

Event / censor **source rows** are recovered afterwards by exact `(RID, DATE)` key-joins against `dxh`.

In [9]:
def derive_survival(anchor_df, dxh_df):
    """01b `derive_survival`, VERBATIM."""
    dxg = {rid: g for rid, g in dxh_df.sort_values("DATE").groupby("RID")}
    recs = []
    for row in anchor_df.itertuples(index=False):
        rid, a = row.RID, row.anchor_date
        g = dxg.get(rid)
        fu = g[g["DATE"] > a] if g is not None else None          # strictly post-anchor (leakage guard)
        sameday_dem = 0
        if g is not None:
            sameday_dem = int(((g["DATE"] == a) & (g["dx_harmonized"] == "Dementia")).sum())
        base = dict(RID=rid, sameday_dementia_at_anchor=sameday_dem)
        if fu is None or len(fu) == 0:
            recs.append({**base, "survival_eligible": False, "no_usable_followup": True,
                         "event_indicator": np.nan, "event_date": pd.NaT, "censor_date": pd.NaT,
                         "followup_end_date": pd.NaT, "time_to_event_or_censor_days": np.nan,
                         "n_post_anchor_visits": 0, "censoring_reason": "no post-anchor diagnosis",
                         "reverted_cn": 0})
            continue
        fu = fu.sort_values("DATE")
        dem = fu[fu["dx_harmonized"] == "Dementia"]
        nondem = fu[fu["dx_harmonized"].isin(["CN", "MCI"])]
        reverted_cn = int((fu["dx_harmonized"] == "CN").any())
        if len(dem):
            edate = dem["DATE"].iloc[0]
            tte = (edate - a).days
            recs.append({**base, "survival_eligible": True, "no_usable_followup": False,
                         "event_indicator": 1, "event_date": edate, "censor_date": pd.NaT,
                         "followup_end_date": edate, "time_to_event_or_censor_days": float(tte),
                         "n_post_anchor_visits": int(len(fu)),
                         "censoring_reason": "", "reverted_cn": reverted_cn})
        elif len(nondem):
            cdate = nondem["DATE"].iloc[-1]
            tte = (cdate - a).days
            recs.append({**base, "survival_eligible": True, "no_usable_followup": False,
                         "event_indicator": 0, "event_date": pd.NaT, "censor_date": cdate,
                         "followup_end_date": cdate, "time_to_event_or_censor_days": float(tte),
                         "n_post_anchor_visits": int(len(fu)),
                         "censoring_reason": "last non-dementia (CN/MCI) follow-up",
                         "reverted_cn": reverted_cn})
        else:
            recs.append({**base, "survival_eligible": False, "no_usable_followup": True,
                         "event_indicator": np.nan, "event_date": pd.NaT, "censor_date": pd.NaT,
                         "followup_end_date": pd.NaT, "time_to_event_or_censor_days": np.nan,
                         "n_post_anchor_visits": int(len(fu)),
                         "censoring_reason": "only unknown/other post-anchor dx", "reverted_cn": 0})
    return pd.DataFrame(recs)


surv = derive_survival(anchor, dxh)

# 01b guard: event/censor must be strictly AFTER the anchor -> demote any non-positive follow-up.
_bad_t = (surv["survival_eligible"] & (surv["time_to_event_or_censor_days"] <= 0)).fillna(False)
surv["nonpositive_followup_flag"] = _bad_t.astype(bool)
if _bad_t.any():
    print("WARNING: non-positive follow-up times:", int(_bad_t.sum()), "-> demoted to no_usable_followup")
    print("  RIDs:", rid_list(surv.loc[_bad_t, "RID"]))
    surv.loc[_bad_t, ["survival_eligible", "no_usable_followup"]] = [False, True]

# --- event / censor source rows (exact key joins; dxh unique on (RID,DATE)) ---
surv = surv.merge(DX_PROV.rename(columns={"DATE": "event_date", "dx_src_row": "event_src_row",
                                          "dx_src_id": "event_src_id"}),
                  on=["RID", "event_date"], how="left", validate="m:1")
surv = surv.merge(DX_PROV.rename(columns={"DATE": "censor_date", "dx_src_row": "censor_src_row",
                                          "dx_src_id": "censor_src_id"}),
                  on=["RID", "censor_date"], how="left", validate="m:1")

n_fu = int(surv["survival_eligible"].sum())
_e = surv[surv["survival_eligible"]]
n_fu_events = int((_e["event_indicator"] == 1).sum())
n_fu_cens = int((_e["event_indicator"] == 0).sum())
print(f"Broad anchor: {len(surv)}")
print(f"Survival-follow-up eligible (B): {n_fu}  |  no usable post-anchor follow-up: "
      f"{int(surv['no_usable_followup'].sum())}")
print(f"Observed dementia events: {n_fu_events}  |  right-censored: {n_fu_cens}")

Broad anchor: 535
Survival-follow-up eligible (B): 410  |  no usable post-anchor follow-up: 125
Observed dementia events: 86  |  right-censored: 324


## 7. Baseline cognition alignment  *(`align_scores` — additive-only change)*

Candidate baseline cognitive variables discovered in Build 0: `CDRSB` (CDR, `CDVERSION ∈ {1,2}`),
`MMSCORE` (MMSE), `MOCA`, `FAQTOTAL` (FAQ). Each is aligned to the anchor with the same **nearest
±90 d** rule used by `01b`.

**Additive-only change:** the function now *also* returns the **matched score date**, from which we
derive `*_offset_days`. The `merge_asof` selection, tolerance and direction are byte-identical to
`01b`, so the selected *values* are unchanged (asserted in the regression cell).

> **Timing caveat carried for the modeling team:** `direction="nearest"` can select a score up to
> **+90 days after** the anchor. Build 1 does **not** change this rule; it now exposes
> `*_offset_days` so Build 3 can audit it. **No cognitive variable is selected here.**

In [10]:
def align_scores(anchor_df, score_df, date_col, vcols, tol_days=ALIGN_TOL_DAYS):
    """01b `align_scores` + ADDITIVE-ONLY provenance: also returns the matched score DATE.
    Selection / tolerance / direction are byte-identical to 01b, so the values are unchanged."""
    s = score_df.copy()
    s["DATE"] = pd.to_datetime(s[date_col], errors="coerce")
    for c in vcols:
        s[c] = pd.to_numeric(s[c], errors="coerce")
    s = (s.dropna(subset=["DATE"]).dropna(subset=vcols, how="all")
          .sort_values("DATE").drop_duplicates(["RID", "DATE"]))
    left = anchor_df[["RID", "anchor_date"]].sort_values("anchor_date")
    out = pd.merge_asof(left, s[["RID", "DATE"] + vcols], left_on="anchor_date", right_on="DATE",
                        by="RID", tolerance=pd.Timedelta(days=tol_days), direction="nearest")
    return out[["RID"] + vcols + ["DATE"]]                 # <-- DATE added (provenance only)


COG = [("cdrsb", "CDRSB"), ("mmse", "MMSCORE"), ("moca", "MOCA"), ("faq", "FAQTOTAL")]
cdr_v = cdr[cdr["CDVERSION"].isin([1, 2])]
_srcs = {"cdrsb": cdr_v, "mmse": mmse, "moca": moca, "faq": faq}

scores = anchor[["RID", "anchor_date"]].copy()
for pfx, vcol in COG:
    part = (align_scores(anchor, _srcs[pfx], "VISDATE", [vcol])
            .rename(columns={"DATE": f"{pfx}_date"}))
    scores = scores.merge(part, on="RID", how="left")
    scores[f"{pfx}_offset_days"] = (scores[f"{pfx}_date"] - scores["anchor_date"]).dt.days

scores = scores.drop(columns=["anchor_date"])
print("Baseline cognition aligned (nearest +/-90d). Availability in the anchor cohort:")
for _, vcol in COG:
    print(f"  {vcol:9s} {int(scores[vcol].notna().sum()):>4d} / {len(scores)}")
print("\nOffset-days summary (negative = score BEFORE anchor, positive = AFTER):")
print(scores[[f"{p}_offset_days" for p, _ in COG]].describe().loc[["min", "50%", "max"]].to_string())

Baseline cognition aligned (nearest +/-90d). Availability in the anchor cohort:
  CDRSB      408 / 535
  MMSCORE    451 / 535
  MOCA       433 / 535
  FAQTOTAL   496 / 535

Offset-days summary (negative = score BEFORE anchor, positive = AFTER):
     cdrsb_offset_days  mmse_offset_days  moca_offset_days  faq_offset_days
min              -89.0             -90.0             -83.0            -83.0
50%              -27.5             -28.0               0.0              0.0
max               83.0              83.0              49.0             88.0


## 8. Master participant-level table and cohort flags

One row per participant. **Feature completeness decides *which model* a participant can enter — it
never influences the anchor.** The primary complete-case rule is exactly `01b`'s `has_primary`:

`primary_eligible = followup_eligible ∧ entry_age ∧ APOE4_COUNT ∧ ptau217`

Secondary features (GFAP, NfL, Aβ42, Aβ40, ratio, cognition) are **never** part of this rule.

In [11]:
plasma_vals = ["ptau217", "abeta42", "abeta40", "nfl", "gfap",
               "ratio_ab42_ab40", "ratio_ab42_ab40_vendor", "n_usable_core_assays"]

anchor_keep = (["RID", "PTID", "anchor_date", "anchor_viscode", "anchor_src_row",
                "anchor_dx_date", "anchor_dx_viscode", "anchor_dx_src_row", "anchor_dx_src_id",
                "align_offset_days", "align_offset_days_abs", "same_day_alignment",
                "anchor_match_type", "anchor_tiebreak_rule", "ptau217_assay"]
               + plasma_vals
               + ["entry_age", "age_src_row", "age_source_date", "entry_age_has_datestamp",
                  "study_entry_date_proxy", "years_entry_to_anchor", "age_at_anchor_approx",
                  "age_is_entry_age_fallback", "age_derivation_suspect",
                  "pre_anchor_dementia_n", "sameday_dementia_dx_n", "qc_dementia_on_or_before_anchor"])

master = (anchor[anchor_keep]
          .merge(apoe_c, on="RID", how="left")
          .merge(scores, on="RID", how="left")
          .merge(surv, on="RID", how="left"))

master["baseline_dx_harmonized"] = "MCI"          # the anchor is MCI by construction
master["time_to_event_or_censor_months"] = master["time_to_event_or_censor_days"] / DAYS_PER_MONTH
master["event_indicator"] = master["event_indicator"].astype("Int64")

# ---- inclusion flags ----
master["broad_anchor_eligible"] = True
master["followup_eligible"] = master["survival_eligible"].fillna(False).astype(bool)
master["entry_age_available"] = master["entry_age"].notna()
master["apoe4_available"] = master["APOE4_COUNT"].notna()
master["ptau217_available"] = master["ptau217"].notna()
master["primary_eligible"] = (master["followup_eligible"] & master["entry_age_available"]
                              & master["apoe4_available"] & master["ptau217_available"])

# ---- secondary availability (INFORMATIONAL ONLY -- never affects primary_eligible) ----
master["gfap_available"] = master["gfap"].notna()
master["nfl_available"] = master["nfl"].notna()
master["amyloid_ratio_available"] = (master["abeta42"].notna() & master["abeta40"].notna()
                                     & master["ratio_ab42_ab40"].notna())

master = master.sort_values("RID").reset_index(drop=True)   # deterministic participant order

n_primary = int(master["primary_eligible"].sum())
n_primary_events = int((master.loc[master["primary_eligible"], "event_indicator"] == 1).sum())
print(f"Master: {master.shape[0]} x {master.shape[1]}")
print(f"A broad anchor       : {n_anchor}")
print(f"B survival follow-up : {n_fu}   (events {n_fu_events}, censored {n_fu_cens})")
print(f"C primary complete   : {n_primary}   (events {n_primary_events})")

Master: 535 x 78
A broad anchor       : 535
B survival follow-up : 410   (events 86, censored 324)
C primary complete   : 401   (events 85)


## 9. Participant-level exclusion log

One row per **broad-anchor** participant (the universe in which downstream attrition happens), so that
**535 → 410 → 401 reconciles exactly**.

Reason precedence (deterministic): a participant who fails at the survival stage is logged there and
is **not** re-assessed for primary-feature missingness.

| `final_inclusion_stage` | meaning |
|---|---|
| `primary_complete_case` | reached cohort C |
| `survival_followup_only` | reached cohort B, dropped before C |
| `broad_anchor_only` | reached cohort A only |

`excl_baseline_dementia` is present in the schema but is **structurally always `False`**: under the
locked v1 rules nobody is *excluded* for on/before-anchor dementia — such cases are **flagged** via
`qc_dementia_on_or_before_anchor` for adjudication (§5b).

In [12]:
m = master

m["excl_nonpositive_followup"] = m["nonpositive_followup_flag"].astype(bool)
m["excl_no_usable_followup"] = (~m["followup_eligible"]) & (~m["excl_nonpositive_followup"])
m["excl_baseline_dementia"] = False        # locked v1: flag-only, never an exclusion (see qc flag)
m["excl_missing_entry_age"] = m["followup_eligible"] & (~m["entry_age_available"])
m["excl_missing_apoe4"] = m["followup_eligible"] & (~m["apoe4_available"])
m["excl_missing_ptau217"] = m["followup_eligible"] & (~m["ptau217_available"])
m["excl_other"] = False


def _stage(r):
    if r["primary_eligible"]:
        return "primary_complete_case"
    return "survival_followup_only" if r["followup_eligible"] else "broad_anchor_only"


def _excl_stage(r):
    if r["primary_eligible"]:
        return ""
    return "primary_complete_case" if r["followup_eligible"] else "survival_followup"


def _reason(r):
    if r["primary_eligible"]:
        return ""
    if not r["followup_eligible"]:
        if r["excl_nonpositive_followup"]:
            return "invalid or non-positive follow-up"
        # granular reason preserved from derive_survival
        return r["censoring_reason"] or "no usable post-anchor follow-up"
    parts = []
    if r["excl_missing_entry_age"]:
        parts.append("missing entry_age")
    if r["excl_missing_apoe4"]:
        parts.append("missing APOE4")
    if r["excl_missing_ptau217"]:
        parts.append("missing p-tau217")
    return "; ".join(parts) if parts else "other algorithmic exclusion"


m["final_inclusion_stage"] = m.apply(_stage, axis=1)
m["exclusion_stage"] = m.apply(_excl_stage, axis=1)
m["exclusion_reason"] = m.apply(_reason, axis=1)

EXCL_COLS = ["excl_no_usable_followup", "excl_nonpositive_followup", "excl_baseline_dementia",
             "excl_missing_entry_age", "excl_missing_apoe4", "excl_missing_ptau217", "excl_other"]

LOG_COLS = (["RID", "PTID", "anchor_date", "final_inclusion_stage", "exclusion_stage", "exclusion_reason",
             "broad_anchor_eligible", "followup_eligible", "primary_eligible"] + EXCL_COLS
            + ["censoring_reason", "n_post_anchor_visits", "event_indicator",
               "entry_age_available", "apoe4_available", "ptau217_available",
               "qc_dementia_on_or_before_anchor", "pre_anchor_dementia_n",
               "sameday_dementia_at_anchor"])

exclusion_log = m[LOG_COLS].sort_values("RID").reset_index(drop=True)

print("final_inclusion_stage:")
print(exclusion_log["final_inclusion_stage"].value_counts().to_string())
print("\nexclusion_reason:")
print(exclusion_log.loc[exclusion_log["exclusion_reason"] != "", "exclusion_reason"]
      .value_counts().to_string())
print("\nexclusion flag totals:")
print(exclusion_log[EXCL_COLS].sum().to_string())

final_inclusion_stage:
final_inclusion_stage
primary_complete_case     401
broad_anchor_only         125
survival_followup_only      9

exclusion_reason:
exclusion_reason
no post-anchor diagnosis    125
missing APOE4                 9

exclusion flag totals:
excl_no_usable_followup      125
excl_nonpositive_followup      0
excl_baseline_dementia         0
excl_missing_entry_age         0
excl_missing_apoe4             9
excl_missing_ptau217           0
excl_other                     0


## 10. Cohort flow table

In [13]:
n_total = int(dxsum["RID"].nunique())
n_mci_any = int(dxh.loc[dxh["dx_harmonized"] == "MCI", "RID"].nunique())
n_plasma_ppl = int(plasma_c.loc[plasma_c["n_usable_core_assays"] >= 1, "RID"].nunique())

n_excl_nofu = int(exclusion_log["excl_no_usable_followup"].sum())
n_excl_nonpos = int(exclusion_log["excl_nonpositive_followup"].sum())
n_excl_primary = int((exclusion_log["exclusion_stage"] == "primary_complete_case").sum())
n_excl_primary_ev = int((m.loc[m["exclusion_stage"] == "primary_complete_case", "event_indicator"] == 1).sum())
n_primary_cens = int((m.loc[m["primary_eligible"], "event_indicator"] == 0).sum())

flow = pd.DataFrame([
    (1,  "Total unique DXSUM participants", n_total, None, "source population"),
    (2,  "Participants with >=1 dated MCI diagnosis", n_mci_any, None, ""),
    (3,  "Participants with >=1 usable plasma draw", n_plasma_ppl, None, ">=1 usable core assay"),
    (4,  "A. Broad MCI plasma-anchor cohort", n_anchor, None,
         "earliest MCI-aligned (+/-90d) plasma draw with >=1 usable core assay"),
    (5,  "Excluded: no usable post-anchor follow-up", n_excl_nofu, None,
         "no valid post-anchor diagnosis -> NOT labelled stable"),
    (6,  "Excluded: invalid or non-positive follow-up", n_excl_nonpos, None,
         "event/censor must be strictly after the anchor"),
    (7,  "B. Survival-follow-up cohort", n_fu, n_fu_events, "A minus no-usable-follow-up"),
    (8,  "   B: observed dementia events", n_fu_events, n_fu_events, "first post-anchor dementia"),
    (9,  "   B: right-censored", n_fu_cens, 0, "censored at last post-anchor non-dementia visit"),
    (10, "Excluded: missing primary feature", n_excl_primary, n_excl_primary_ev,
         "entry_age / APOE4 / p-tau217"),
    (11, "C. Primary complete-case cohort", n_primary, n_primary_events,
         "B + entry_age + APOE4 + p-tau217"),
    (12, "   C: observed dementia events", n_primary_events, n_primary_events, ""),
    (13, "   C: right-censored", n_primary_cens, 0, ""),
], columns=["stage_order", "step", "n", "n_events", "notes"])

print(flow.to_string(index=False))

 stage_order                                        step    n  n_events                                                                notes
           1             Total unique DXSUM participants 3789       NaN                                                    source population
           2   Participants with >=1 dated MCI diagnosis 1801       NaN                                                                     
           3    Participants with >=1 usable plasma draw 1615       NaN                                                >=1 usable core assay
           4           A. Broad MCI plasma-anchor cohort  535       NaN earliest MCI-aligned (+/-90d) plasma draw with >=1 usable core assay
           5   Excluded: no usable post-anchor follow-up  125       NaN                no valid post-anchor diagnosis -> NOT labelled stable
           6 Excluded: invalid or non-positive follow-up    0       NaN                       event/censor must be strictly after the anchor
           7 

## 11. Frozen cohort tables

Three **nested** participant-level tables sharing one schema, sorted deterministically by `RID`.
Dates are serialised as `YYYY-MM-DD`. Raw values are preserved; nothing is transformed or imputed.

`ptau181` is **absent by design** — see the Build 1 report.

In [14]:
COLS = [
    # --- identification & anchor ---
    "RID", "PTID",
    "anchor_date", "anchor_viscode", "anchor_src_row",
    "anchor_dx_date", "anchor_dx_viscode", "anchor_dx_src_row", "anchor_dx_src_id",
    "align_offset_days", "align_offset_days_abs", "same_day_alignment", "anchor_match_type",
    "baseline_dx_harmonized", "n_usable_core_assays", "anchor_tiebreak_rule",
    # --- baseline covariates (PRIMARY) ---
    "entry_age", "age_src_row", "age_source_date", "entry_age_has_datestamp",
    "study_entry_date_proxy", "years_entry_to_anchor", "age_at_anchor_approx",
    "age_is_entry_age_fallback", "age_derivation_suspect",
    "GENOTYPE", "APOE4_COUNT", "apoe_src_row", "apoe_src_id",
    "ptau217", "ptau217_assay",
    # --- secondary raw features (never affect primary inclusion) ---
    "gfap", "nfl", "abeta42", "abeta40", "ratio_ab42_ab40", "ratio_ab42_ab40_vendor",
    "CDRSB", "cdrsb_date", "cdrsb_offset_days",
    "MMSCORE", "mmse_date", "mmse_offset_days",
    "MOCA", "moca_date", "moca_offset_days",
    "FAQTOTAL", "faq_date", "faq_offset_days",
    # --- outcome ---
    "event_indicator", "event_date", "event_src_row", "event_src_id",
    "censor_date", "censor_src_row", "censor_src_id",
    "followup_end_date", "time_to_event_or_censor_days", "time_to_event_or_censor_months",
    "n_post_anchor_visits", "censoring_reason", "reverted_cn",
    # --- inclusion flags ---
    "broad_anchor_eligible", "followup_eligible", "primary_eligible",
    "final_inclusion_stage", "exclusion_stage", "exclusion_reason",
    # --- availability flags ---
    "entry_age_available", "apoe4_available", "ptau217_available",
    "gfap_available", "nfl_available", "amyloid_ratio_available",
    # --- QC flags ---
    "sameday_dementia_at_anchor", "pre_anchor_dementia_n", "qc_dementia_on_or_before_anchor",
    "nonpositive_followup_flag",
]

DATE_COLS = ["anchor_date", "anchor_dx_date", "age_source_date", "study_entry_date_proxy",
             "cdrsb_date", "mmse_date", "moca_date", "faq_date",
             "event_date", "censor_date", "followup_end_date"]

missing_cols = [c for c in COLS if c not in m.columns]
check("frozen schema: every declared column exists in the master", not missing_cols, str(missing_cols))


def freeze(df: pd.DataFrame) -> pd.DataFrame:
    out = df[COLS].sort_values("RID").reset_index(drop=True).copy()
    for c in DATE_COLS:
        out[c] = pd.to_datetime(out[c]).dt.strftime("%Y-%m-%d")
    return out


anchor_tbl = freeze(m)
followup_tbl = freeze(m[m["followup_eligible"]])
primary_tbl = freeze(m[m["primary_eligible"]])

print(f"anchor_tbl   : {anchor_tbl.shape}")
print(f"followup_tbl : {followup_tbl.shape}")
print(f"primary_tbl  : {primary_tbl.shape}")

  PASS  frozen schema: every declared column exists in the master  [[]]
anchor_tbl   : (535, 78)
followup_tbl : (410, 78)
primary_tbl  : (401, 78)


## 12. Validation — full assertion battery

Every check in the master spec (§Build 1 "Required logic checks" 1–15) plus the explicitly requested
verifications. Everything is **recomputed from source**, never hard-coded. Independent recomputations
deliberately use a *different code path* from the canonical functions so they are a real test.

In [15]:
# ---------- 1-3. one row per participant, no missing / duplicate IDs ----------
for nm, t in [("anchor", anchor_tbl), ("followup", followup_tbl), ("primary", primary_tbl),
              ("exclusion_log", exclusion_log)]:
    check(f"[{nm}] one row per participant / no duplicate RID",
          not t["RID"].duplicated().any(),
          rid_list(t.loc[t["RID"].duplicated(), "RID"]))
    check(f"[{nm}] no missing participant ID", t["RID"].notna().all())
    check(f"[{nm}] sorted deterministically by RID", t["RID"].is_monotonic_increasing)

# ---------- 4. all anchor diagnoses are MCI (verified against dxh, not the flag) ----------
_ver = anchor_tbl[["RID", "anchor_dx_date"]].copy()
_ver["anchor_dx_date"] = pd.to_datetime(_ver["anchor_dx_date"])
_ver = _ver.merge(dxh[["RID", "DATE", "dx_harmonized"]].rename(columns={"DATE": "anchor_dx_date"}),
                  on=["RID", "anchor_dx_date"], how="left")
_bad = _ver[_ver["dx_harmonized"] != "MCI"]
check("all anchor-matched diagnoses are MCI (re-read from dxh)", _bad.empty, rid_list(_bad["RID"]))
check("baseline_dx_harmonized == 'MCI' for every row",
      (anchor_tbl["baseline_dx_harmonized"] == "MCI").all())

# ---------- 5. all anchor matches within +/-90 absolute days ----------
_off = anchor_tbl["align_offset_days_abs"]
check(f"all anchor matches within +/-{ALIGN_TOL_DAYS} days (max={int(_off.max())}d)",
      bool((_off <= ALIGN_TOL_DAYS).all()) and bool(_off.notna().all()),
      rid_list(anchor_tbl.loc[~(_off <= ALIGN_TOL_DAYS), "RID"]))

# ---------- 6. anchor not selected using future completeness or outcome ----------
# Independent recomputation of every ELIGIBLE candidate draw, then take the earliest per participant.
_cand = pd.merge_asof(
    plasma_c[plasma_c["n_usable_core_assays"] >= 1].dropna(subset=["DATE"]).sort_values("DATE"),
    (dxh[["RID", "DATE", "dx_harmonized"]].rename(columns={"DATE": "dx_date"})
        .dropna(subset=["dx_date"]).sort_values("dx_date")),
    left_on="DATE", right_on="dx_date", by="RID",
    tolerance=pd.Timedelta(days=ALIGN_TOL_DAYS), direction="nearest")
_cand = _cand[_cand["dx_harmonized"] == "MCI"]
_earliest = _cand.groupby("RID")["DATE"].min().rename("earliest_candidate")

_cmp = (anchor_tbl[["RID", "anchor_date"]].assign(anchor_date=lambda d: pd.to_datetime(d["anchor_date"]))
        .merge(_earliest, on="RID", how="left"))
_mismatch = _cmp[_cmp["anchor_date"] != _cmp["earliest_candidate"]]
check("anchor == EARLIEST eligible MCI-aligned draw for every participant "
      "(independent recomputation; no future info can enter)",
      _mismatch.empty, rid_list(_mismatch["RID"]))
check("candidate universe reproduces the anchor cohort exactly",
      set(_earliest.index) == set(anchor_tbl["RID"]))

# Positive evidence: later, MORE-COMPLETE draws exist and were NOT preferred.
_rich = (_cand.merge(anchor_tbl[["RID", "anchor_date", "n_usable_core_assays"]]
                     .assign(anchor_date=lambda d: pd.to_datetime(d["anchor_date"]))
                     .rename(columns={"n_usable_core_assays": "anchor_n_assays"}), on="RID"))
_later_richer = _rich[(_rich["DATE"] > _rich["anchor_date"])
                      & (_rich["n_usable_core_assays"] > _rich["anchor_n_assays"])]
check("later, more-complete draws were NOT preferred over the earliest draw",
      True, f"{_later_richer['RID'].nunique()} participants had a later draw with MORE usable assays; "
            f"the earliest draw was still anchored in every case")

  PASS  [anchor] one row per participant / no duplicate RID  [n=0 RIDs=[]]
  PASS  [anchor] no missing participant ID
  PASS  [anchor] sorted deterministically by RID
  PASS  [followup] one row per participant / no duplicate RID  [n=0 RIDs=[]]
  PASS  [followup] no missing participant ID
  PASS  [followup] sorted deterministically by RID
  PASS  [primary] one row per participant / no duplicate RID  [n=0 RIDs=[]]
  PASS  [primary] no missing participant ID
  PASS  [primary] sorted deterministically by RID
  PASS  [exclusion_log] one row per participant / no duplicate RID  [n=0 RIDs=[]]
  PASS  [exclusion_log] no missing participant ID
  PASS  [exclusion_log] sorted deterministically by RID
  PASS  all anchor-matched diagnoses are MCI (re-read from dxh)  [n=0 RIDs=[]]
  PASS  baseline_dx_harmonized == 'MCI' for every row
  PASS  all anchor matches within +/-90 days (max=90d)  [n=0 RIDs=[]]
  PASS  anchor == EARLIEST eligible MCI-aligned draw for every participant (independent recomputati

In [16]:
# ---------- 7-8. survival cohort has an outcome date; follow-up strictly positive ----------
_f = followup_tbl.copy()
_no_end = _f[_f["followup_end_date"].isna()]
check("every survival participant has an event or censoring date", _no_end.empty, rid_list(_no_end["RID"]))
_nonpos = _f[~(_f["time_to_event_or_censor_days"] > 0)]
check(f"all follow-up durations strictly positive (min={_f['time_to_event_or_censor_days'].min():.0f}d)",
      _nonpos.empty, rid_list(_nonpos["RID"]))
check("event/censor dates are strictly AFTER the anchor",
      bool((pd.to_datetime(_f["followup_end_date"]) > pd.to_datetime(_f["anchor_date"])).all()))

# ---------- 9. events are the FIRST qualifying post-anchor dementia (independent recompute) ----------
_a = _f[["RID", "anchor_date"]].assign(anchor_date=lambda d: pd.to_datetime(d["anchor_date"]))
_dm = _a.merge(dxh.loc[dxh["dx_harmonized"] == "Dementia", ["RID", "DATE"]], on="RID", how="left")
_dm = _dm[_dm["DATE"] > _dm["anchor_date"]]
_first_dem = _dm.groupby("RID")["DATE"].min().rename("first_post_anchor_dementia")

_ev = _f[_f["event_indicator"] == 1][["RID", "event_date"]].assign(
    event_date=lambda d: pd.to_datetime(d["event_date"]))
_ev = _ev.merge(_first_dem, on="RID", how="left")
_bad_ev = _ev[_ev["event_date"] != _ev["first_post_anchor_dementia"]]
check("every event date IS the first post-anchor dementia diagnosis (independent recomputation)",
      _bad_ev.empty, rid_list(_bad_ev["RID"]))

_cens_ids = set(_f.loc[_f["event_indicator"] == 0, "RID"])
_leak = _cens_ids & set(_first_dem.index)
check("no censored participant has ANY post-anchor dementia diagnosis", not _leak, rid_list(_leak))

# ---------- 10. participants with events are not censored before the event ----------
_evrows = _f[_f["event_indicator"] == 1]
check("event participants carry no censor date (never censored before their event)",
      _evrows["censor_date"].isna().all(), rid_list(_evrows.loc[_evrows["censor_date"].notna(), "RID"]))
check("event participants: followup_end_date == event_date",
      (_evrows["followup_end_date"] == _evrows["event_date"]).all())
check("every event row is traceable to a DXSUM source row",
      _evrows["event_src_row"].notna().all(), rid_list(_evrows.loc[_evrows["event_src_row"].isna(), "RID"]))

# ---------- 11. censored participants have a valid LAST post-anchor non-dementia visit ----------
_nd = _a.merge(dxh.loc[dxh["dx_harmonized"].isin(["CN", "MCI"]), ["RID", "DATE"]], on="RID", how="left")
_nd = _nd[_nd["DATE"] > _nd["anchor_date"]]
_last_nd = _nd.groupby("RID")["DATE"].max().rename("last_post_anchor_nondementia")

_cs = _f[_f["event_indicator"] == 0][["RID", "censor_date"]].assign(
    censor_date=lambda d: pd.to_datetime(d["censor_date"]))
_cs = _cs.merge(_last_nd, on="RID", how="left")
_bad_cs = _cs[(_cs["censor_date"].isna()) | (_cs["censor_date"] != _cs["last_post_anchor_nondementia"])]
check("every censored participant is censored at their LAST post-anchor non-dementia visit "
      "(independent recomputation)", _bad_cs.empty, rid_list(_bad_cs["RID"]))

_censrows = _f[_f["event_indicator"] == 0]
check("every censor row is traceable to a DXSUM source row",
      _censrows["censor_src_row"].notna().all(),
      rid_list(_censrows.loc[_censrows["censor_src_row"].isna(), "RID"]))

# the censoring visit really is CN/MCI in the source
_cv = _cs[["RID", "censor_date"]].merge(
    dxh[["RID", "DATE", "dx_harmonized"]].rename(columns={"DATE": "censor_date"}),
    on=["RID", "censor_date"], how="left")
check("every censoring diagnosis is CN or MCI in the source",
      _cv["dx_harmonized"].isin(["CN", "MCI"]).all(),
      rid_list(_cv.loc[~_cv["dx_harmonized"].isin(["CN", "MCI"]), "RID"]))

  PASS  every survival participant has an event or censoring date  [n=0 RIDs=[]]
  PASS  all follow-up durations strictly positive (min=1d)  [n=0 RIDs=[]]
  PASS  event/censor dates are strictly AFTER the anchor
  PASS  every event date IS the first post-anchor dementia diagnosis (independent recomputation)  [n=0 RIDs=[]]
  PASS  no censored participant has ANY post-anchor dementia diagnosis  [n=0 RIDs=[]]
  PASS  event participants carry no censor date (never censored before their event)  [n=0 RIDs=[]]
  PASS  event participants: followup_end_date == event_date
  PASS  every event row is traceable to a DXSUM source row  [n=0 RIDs=[]]
  PASS  every censored participant is censored at their LAST post-anchor non-dementia visit (independent recomputation)  [n=0 RIDs=[]]
  PASS  every censor row is traceable to a DXSUM source row  [n=0 RIDs=[]]
  PASS  every censoring diagnosis is CN or MCI in the source  [n=0 RIDs=[]]


In [17]:
# ---------- 12. no-usable-follow-up participants are EXCLUDED, never labelled stable ----------
_nofu = m[~m["followup_eligible"]]
check("no-usable-follow-up participants are excluded from the survival cohort",
      not (set(_nofu["RID"]) & set(followup_tbl["RID"])))
check("no-usable-follow-up participants have NO event_indicator (not silently coded as censored/stable)",
      _nofu["event_indicator"].isna().all(),
      rid_list(_nofu.loc[_nofu["event_indicator"].notna(), "RID"]))
check("no-usable-follow-up participants have no follow-up time",
      _nofu["time_to_event_or_censor_days"].isna().all())

# ---------- 13. primary cohort completeness ----------
for col in ["entry_age", "APOE4_COUNT", "ptau217"]:
    bad = primary_tbl[primary_tbl[col].isna()]
    check(f"[primary] {col} is nonmissing for every participant", bad.empty, rid_list(bad["RID"]))

# ---------- 14. secondary-feature missingness does NOT affect primary inclusion ----------
_recomputed = (m["followup_eligible"] & m["entry_age"].notna()
               & m["APOE4_COUNT"].notna() & m["ptau217"].notna())
check("primary_eligible depends ONLY on followup + entry_age + APOE4 + p-tau217",
      bool((_recomputed == m["primary_eligible"]).all()))

_sec = {"gfap": int(primary_tbl["gfap"].isna().sum()),
        "nfl": int(primary_tbl["nfl"].isna().sum()),
        "abeta42": int(primary_tbl["abeta42"].isna().sum()),
        "abeta40": int(primary_tbl["abeta40"].isna().sum()),
        "ratio_ab42_ab40": int(primary_tbl["ratio_ab42_ab40"].isna().sum()),
        "CDRSB": int(primary_tbl["CDRSB"].isna().sum()),
        "MMSCORE": int(primary_tbl["MMSCORE"].isna().sum()),
        "MOCA": int(primary_tbl["MOCA"].isna().sum()),
        "FAQTOTAL": int(primary_tbl["FAQTOTAL"].isna().sum())}
check("participants WITH missing secondary features are still retained in the primary cohort",
      sum(_sec.values()) > 0, f"missing counts inside the primary cohort: {_sec}")

# ---------- nesting ----------
check("cohorts are strictly nested: primary subset-of followup subset-of anchor",
      set(primary_tbl["RID"]) <= set(followup_tbl["RID"]) <= set(anchor_tbl["RID"]))

# ---------- source-row provenance completeness ----------
check("every anchor row is traceable to a plasma source row",
      anchor_tbl["anchor_src_row"].notna().all())
check("every matched MCI diagnosis is traceable to a DXSUM source row",
      anchor_tbl["anchor_dx_src_row"].notna().all())
check("age source row present wherever entry_age is present",
      anchor_tbl.loc[anchor_tbl["entry_age"].notna(), "age_src_row"].notna().all())
check("APOE source row present wherever APOE4_COUNT is present",
      anchor_tbl.loc[anchor_tbl["APOE4_COUNT"].notna(), "apoe_src_row"].notna().all())
check("age_source_date is structurally absent (My_Table carries no datestamp)",
      anchor_tbl["age_source_date"].isna().all()
      and not bool(anchor_tbl["entry_age_has_datestamp"].any()))

  PASS  no-usable-follow-up participants are excluded from the survival cohort
  PASS  no-usable-follow-up participants have NO event_indicator (not silently coded as censored/stable)  [n=0 RIDs=[]]
  PASS  no-usable-follow-up participants have no follow-up time
  PASS  [primary] entry_age is nonmissing for every participant  [n=0 RIDs=[]]
  PASS  [primary] APOE4_COUNT is nonmissing for every participant  [n=0 RIDs=[]]
  PASS  [primary] ptau217 is nonmissing for every participant  [n=0 RIDs=[]]
  PASS  primary_eligible depends ONLY on followup + entry_age + APOE4 + p-tau217
  PASS  participants WITH missing secondary features are still retained in the primary cohort  [missing counts inside the primary cohort: {'gfap': 40, 'nfl': 40, 'abeta42': 2, 'abeta40': 2, 'ratio_ab42_ab40': 2, 'CDRSB': 87, 'MMSCORE': 47, 'MOCA': 88, 'FAQTOTAL': 31}]
  PASS  cohorts are strictly nested: primary subset-of followup subset-of anchor
  PASS  every anchor row is traceable to a plasma source row
  PASS  

In [18]:
# ---------- 15. cohort flow reconciles EXACTLY with the exclusion log ----------
log_primary = int((exclusion_log["final_inclusion_stage"] == "primary_complete_case").sum())
log_fu_only = int((exclusion_log["final_inclusion_stage"] == "survival_followup_only").sum())
log_anchor_only = int((exclusion_log["final_inclusion_stage"] == "broad_anchor_only").sum())

check("exclusion log covers exactly the broad-anchor cohort",
      len(exclusion_log) == n_anchor == len(anchor_tbl))
check("log stages partition the anchor cohort (primary + followup_only + anchor_only == A)",
      log_primary + log_fu_only + log_anchor_only == n_anchor,
      f"{log_primary} + {log_fu_only} + {log_anchor_only} == {n_anchor}")
check("log reproduces the survival-follow-up cohort (B)",
      log_primary + log_fu_only == n_fu == len(followup_tbl),
      f"{log_primary} + {log_fu_only} == {n_fu}")
check("log reproduces the primary complete-case cohort (C)",
      log_primary == n_primary == len(primary_tbl), f"{log_primary} == {n_primary}")
check("every excluded participant has a non-empty exclusion reason",
      bool((exclusion_log.loc[exclusion_log["exclusion_stage"] != "", "exclusion_reason"] != "").all()))
check("every included participant has an empty exclusion stage and reason",
      bool((exclusion_log.loc[exclusion_log["exclusion_stage"] == "", "exclusion_reason"] == "").all()))

_flow_A = int(flow.loc[flow["step"] == "A. Broad MCI plasma-anchor cohort", "n"].iloc[0])
_flow_B = int(flow.loc[flow["step"] == "B. Survival-follow-up cohort", "n"].iloc[0])
_flow_C = int(flow.loc[flow["step"] == "C. Primary complete-case cohort", "n"].iloc[0])
_flow_nofu = int(flow.loc[flow["step"] == "Excluded: no usable post-anchor follow-up", "n"].iloc[0])
_flow_nonpos = int(flow.loc[flow["step"] == "Excluded: invalid or non-positive follow-up", "n"].iloc[0])
_flow_exprim = int(flow.loc[flow["step"] == "Excluded: missing primary feature", "n"].iloc[0])

check("flow: A - (no-followup + non-positive-followup) == B",
      _flow_A - (_flow_nofu + _flow_nonpos) == _flow_B,
      f"{_flow_A} - ({_flow_nofu} + {_flow_nonpos}) == {_flow_B}")
check("flow: B - excluded-missing-primary-feature == C",
      _flow_B - _flow_exprim == _flow_C, f"{_flow_B} - {_flow_exprim} == {_flow_C}")
check("flow counts equal the frozen table row counts",
      (_flow_A, _flow_B, _flow_C) == (len(anchor_tbl), len(followup_tbl), len(primary_tbl)))
check("flow event counts equal the frozen table event counts",
      int(flow.loc[flow["step"] == "B. Survival-follow-up cohort", "n_events"].iloc[0])
      == int((followup_tbl["event_indicator"] == 1).sum())
      and int(flow.loc[flow["step"] == "C. Primary complete-case cohort", "n_events"].iloc[0])
      == int((primary_tbl["event_indicator"] == 1).sum()))

  PASS  exclusion log covers exactly the broad-anchor cohort
  PASS  log stages partition the anchor cohort (primary + followup_only + anchor_only == A)  [401 + 9 + 125 == 535]
  PASS  log reproduces the survival-follow-up cohort (B)  [401 + 9 == 410]
  PASS  log reproduces the primary complete-case cohort (C)  [401 == 401]
  PASS  every excluded participant has a non-empty exclusion reason
  PASS  every included participant has an empty exclusion stage and reason
  PASS  flow: A - (no-followup + non-positive-followup) == B  [535 - (125 + 0) == 410]
  PASS  flow: B - excluded-missing-primary-feature == C  [410 - 9 == 401]
  PASS  flow counts equal the frozen table row counts
  PASS  flow event counts equal the frozen table event counts


In [19]:
# ---------- explicitly requested verifications (RECOMPUTED, never hard-coded) ----------
_fu = m[m["followup_eligible"]]

_miss_age = _fu[_fu["entry_age"].isna()]
_miss_p217 = _fu[_fu["ptau217"].isna()]
_miss_apoe = _fu[_fu["APOE4_COUNT"].isna()]

check("entry_age has ZERO missingness in the survival cohort (Build 0 finding)",
      _miss_age.empty, f"missing n={len(_miss_age)}")
check("p-tau217 has ZERO missingness in the survival cohort (Build 0 finding)",
      _miss_p217.empty, f"missing n={len(_miss_p217)}")

_drop = n_fu - n_primary
check("the B->C drop is explained ENTIRELY by missing APOE4",
      len(_miss_apoe) == _drop,
      f"B={n_fu}, C={n_primary}, drop={_drop}, APOE4-missing={len(_miss_apoe)}; {rid_list(_miss_apoe['RID'])}")

_apoe_events = int((_miss_apoe["event_indicator"] == 1).sum())
check("exactly one of the APOE4-missing survival participants is an event",
      _apoe_events == (n_fu_events - n_primary_events),
      f"APOE4-missing events={_apoe_events}; events B={n_fu_events} -> C={n_primary_events}; "
      f"event RID(s)={rid_list(_miss_apoe.loc[_miss_apoe['event_indicator'] == 1, 'RID'])}")

print(f"\n  APOE4-missing survival participants ({len(_miss_apoe)}): "
      f"{sorted(int(r) for r in _miss_apoe['RID'])}")
print(f"  of which events ({_apoe_events}): "
      f"{sorted(int(r) for r in _miss_apoe.loc[_miss_apoe['event_indicator'] == 1, 'RID'])}")

  PASS  entry_age has ZERO missingness in the survival cohort (Build 0 finding)  [missing n=0]
  PASS  p-tau217 has ZERO missingness in the survival cohort (Build 0 finding)  [missing n=0]
  PASS  the B->C drop is explained ENTIRELY by missing APOE4  [B=410, C=401, drop=9, APOE4-missing=9; n=9 RIDs=[7079, 10167, 10193, 10251, 10276, 10432, 10441, 10452, 10672]]
  PASS  exactly one of the APOE4-missing survival participants is an event  [APOE4-missing events=1; events B=86 -> C=85; event RID(s)=n=1 RIDs=[10251]]

  APOE4-missing survival participants (9): [7079, 10167, 10193, 10251, 10276, 10432, 10441, 10452, 10672]
  of which events (1): [10251]


## 13. Regression check against the `01b` audit  *(read-only)*

Proves the freeze **reproduces the validated audit exactly**, rather than being an independent second
implementation. Compares every shared scientific column participant-by-participant.
`01b`'s artifact is opened **read-only** and is never modified.

In [20]:
PRIOR = PRIOR_DIR / "mci_survival_anchor_cohort_raw.csv"
if PRIOR.exists():
    prior = pd.read_csv(PRIOR)
    check("[regression] identical participant set vs 01b",
          set(prior["RID"]) == set(anchor_tbl["RID"]),
          f"01b n={len(prior)}, 01c n={len(anchor_tbl)}; "
          f"only-01b={rid_list(set(prior['RID']) - set(anchor_tbl['RID']))}; "
          f"only-01c={rid_list(set(anchor_tbl['RID']) - set(prior['RID']))}")

    a = anchor_tbl.set_index("RID").sort_index()
    b = prior.set_index("RID").sort_index()

    NUM = ["align_offset_days", "ptau217", "abeta42", "abeta40", "nfl", "gfap",
           "ratio_ab42_ab40", "ratio_ab42_ab40_vendor", "n_usable_core_assays",
           "entry_age", "APOE4_COUNT", "CDRSB", "MMSCORE", "MOCA", "FAQTOTAL",
           "time_to_event_or_censor_days", "n_post_anchor_visits"]
    DAT = ["anchor_date", "anchor_dx_date", "event_date", "censor_date", "followup_end_date"]

    diffs = {}
    for c in NUM:
        x, y = a[c].astype(float), b[c].astype(float)
        bad = ~((x - y).abs() <= 1e-9) & ~(x.isna() & y.isna())
        if bad.any():
            diffs[c] = rid_list(a.index[bad])
    for c in DAT:
        x, y = pd.to_datetime(a[c]), pd.to_datetime(b[c])
        bad = (x != y) & ~(x.isna() & y.isna())
        if bad.any():
            diffs[c] = rid_list(a.index[bad])

    x, y = a["event_indicator"].astype("Float64"), b["event_indicator"].astype("Float64")
    bad = (x != y) & ~(x.isna() & y.isna())
    if bad.any():
        diffs["event_indicator"] = rid_list(a.index[bad])

    bad = a["followup_eligible"].astype(bool) != b["survival_followup_eligible_flag"].astype(bool)
    if bad.any():
        diffs["followup_eligible"] = rid_list(a.index[bad])
    bad = a["primary_eligible"].astype(bool) != b["has_primary"].astype(bool)
    if bad.any():
        diffs["primary_eligible"] = rid_list(a.index[bad])

    check("[regression] 01c reproduces 01b EXACTLY on every shared scientific column "
          f"({len(NUM) + len(DAT) + 3} columns x {len(a)} participants)",
          not diffs, json.dumps(diffs, indent=2) if diffs else "0 differing cells")
else:
    check("[regression] 01b artifact present", False, f"missing: {PRIOR}")

  PASS  [regression] identical participant set vs 01b  [01b n=535, 01c n=535; only-01b=n=0 RIDs=[]; only-01c=n=0 RIDs=[]]
  PASS  [regression] 01c reproduces 01b EXACTLY on every shared scientific column (25 columns x 535 participants)  [0 differing cells]


## 14. Reconciliation against the audit landmarks

Targets are **reported, never forced**. A mismatch triggers a discrepancy report and the cohort is
**not** declared frozen.

In [21]:
recon = pd.DataFrame([
    ("A. Broad anchor", n_anchor, None, TARGETS["anchor"], None),
    ("B. Survival-follow-up", n_fu, n_fu_events, TARGETS["followup"], TARGETS["followup_events"]),
    ("C. Primary complete-case", n_primary, n_primary_events, TARGETS["primary"], TARGETS["primary_events"]),
], columns=["stage", "n", "events", "target_n", "target_events"])
recon["diff_n"] = recon["n"] - recon["target_n"]
recon["diff_events"] = recon["events"] - recon["target_events"]
print(recon.to_string(index=False))

DISCREPANT = bool((recon["diff_n"].fillna(0) != 0).any() or (recon["diff_events"].fillna(0) != 0).any())
check("cohort counts reconcile with the 01b audit landmarks (535 / 410 / 86 / 401 / 85)",
      not DISCREPANT, recon.to_string(index=False))
print("\nFREEZE STATUS:", "DISCREPANT -- NOT FROZEN" if DISCREPANT else "RECONCILED -- SAFE TO FREEZE")

                   stage   n  events  target_n  target_events  diff_n  diff_events
         A. Broad anchor 535     NaN       535            NaN       0          NaN
   B. Survival-follow-up 410    86.0       410           86.0       0          0.0
C. Primary complete-case 401    85.0       401           85.0       0          0.0
  PASS  cohort counts reconcile with the 01b audit landmarks (535 / 410 / 86 / 401 / 85)  [                   stage   n  events  target_n  target_events  diff_n  diff_events
         A. Broad anchor 535     NaN       535            NaN       0          NaN
   B. Survival-follow-up 410    86.0       410           86.0       0          0.0
C. Primary complete-case 401    85.0       401           85.0       0          0.0]

FREEZE STATUS: RECONCILED -- SAFE TO FREEZE


## 15. Write frozen outputs + provenance manifest

In [22]:
OUTPUTS = {
    f"mci_survival_anchor_cohort_{VERSION}.tsv": anchor_tbl,
    f"mci_survival_followup_cohort_{VERSION}.tsv": followup_tbl,
    f"mci_survival_primary_cohort_{VERSION}.tsv": primary_tbl,
    f"mci_survival_exclusion_log_{VERSION}.tsv": exclusion_log,
    f"mci_survival_cohort_flow_{VERSION}.tsv": flow,
}

# Guard: never overwrite a prior audit artifact.
check("no prior (01b) artifact is written by this notebook",
      all(not (PRIOR_DIR / n).exists() for n in OUTPUTS))

paths = {name: save_tsv(df, name) for name, df in OUTPUTS.items()}


def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=PROJECT_ROOT,
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None


manifest = {
    "build": 1,
    "version": VERSION,
    "notebook": "notebooks/01c_mci_survival_cohort_freeze.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git": {"commit": git("rev-parse", "HEAD"), "dirty": bool(git("status", "--porcelain"))},
    "parameters": {
        "align_tolerance_days": ALIGN_TOL_DAYS,
        "missing_sentinels": MISSING_SENTINELS,
        "nonpositive_assay_is_missing": NONPOSITIVE_IS_MISSING,
        "core_assays": CORE_ASSAYS,
        "days_per_month": DAYS_PER_MONTH,
        "days_per_year": DAYS_PER_YEAR,
        "random_seed": RANDOM_SEED,
        "imputation": "none",
    },
    "locked_rules": {
        "diagnosis_coding": "1=CN, 2=MCI, 3=Dementia; blank=unknown/other (never event, never censor)",
        "same_day_diagnosis_conflict": "collapse per (RID,DATE) keeping highest severity",
        "anchor": "earliest plasma draw with >=1 usable core assay, nearest MCI dx within +/-90d",
        "anchor_tiebreak": ("earliest plasma DATE; plasma pre-deduplicated to one row per (RID,DATE) "
                            "keep=first; stable original row order"),
        "time_origin": "plasma anchor date",
        "event": "first dementia diagnosis strictly after the anchor",
        "censoring": "last post-anchor non-dementia (CN/MCI) diagnosis date",
        "no_usable_followup": "excluded from the survival cohort; never labelled stable",
        "primary_complete_case": "survival-eligible + entry_age + APOE4_COUNT + ptau217",
        "age": "entry_age is authoritative and UNDATED; age-at-anchor is not derived",
        "dementia_on_or_before_anchor": "flagged for adjudication, not excluded (locked v1 rules)",
    },
    "source_row_id_rule": ("<prefix>_src_row = 0-based positional index of the record in the raw CSV "
                           "as delivered (row 0 = first data row after the header), assigned "
                           "immediately after pd.read_csv and BEFORE any filtering/sorting/dedup. "
                           "Native ADNI ids carried where present: dx_src_id (DXSUM.ID), "
                           "apoe_src_id (APOERES.ID). The UPENN plasma panel has NO native row id."),
    "sources": [
        {"file": f"Data/raw/{fn}", "sha256": sha256(RAW_DIR / fn),
         "mtime_utc": datetime.fromtimestamp((RAW_DIR / fn).stat().st_mtime, timezone.utc).isoformat(timespec="seconds"),
         "bytes": (RAW_DIR / fn).stat().st_size}
        for fn in SRC_FILES.values()
    ],
    "counts": {
        "dxsum_participants": n_total,
        "participants_with_mci_dx": n_mci_any,
        "participants_with_usable_plasma": n_plasma_ppl,
        "anchor_cohort": n_anchor,
        "followup_cohort": n_fu,
        "followup_events": n_fu_events,
        "followup_censored": n_fu_cens,
        "primary_cohort": n_primary,
        "primary_events": n_primary_events,
        "excluded_no_usable_followup": n_excl_nofu,
        "excluded_nonpositive_followup": n_excl_nonpos,
        "excluded_missing_primary_feature": n_excl_primary,
        "flagged_dementia_on_or_before_anchor": n_pre_dem,
    },
    "reconciliation_targets": TARGETS,
    "reconciled": not DISCREPANT,
    "outputs": [
        {"file": f"outputs/01c_mci_survival_cohort_freeze/{n}", "sha256": sha256(p),
         "n_rows": int(OUTPUTS[n].shape[0]), "n_cols": int(OUTPUTS[n].shape[1])}
        for n, p in paths.items()
    ],
    "notes": {"ptau181": PTAU181_STATUS,
              "censoring_limitation": ("no death / loss-to-follow-up table exists; censoring is "
                                       "administrative (last clinical visit) and may be informative"),
              "cognition_timing": ("baseline cognition uses nearest +/-90d and may fall AFTER the "
                                   "anchor; *_offset_days is carried so Build 3 can audit it")},
    "assertions": {"n_total": len(ASSERTIONS),
                   "n_passed": int(sum(a["passed"] for a in ASSERTIONS)),
                   "n_failed": int(sum(not a["passed"] for a in ASSERTIONS)),
                   "results": ASSERTIONS},
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "numpy": np.__version__, "platform": platform.platform()},
}

MANIFEST_PATH = OUT_DIR / f"mci_survival_freeze_provenance_{VERSION}.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=str))
print(f"  saved -> {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")

  PASS  no prior (01b) artifact is written by this notebook
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_anchor_cohort_v1.tsv   (535 x 78)


  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_followup_cohort_v1.tsv   (410 x 78)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_primary_cohort_v1.tsv   (401 x 78)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_exclusion_log_v1.tsv   (535 x 25)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_cohort_flow_v1.tsv   (13 x 5)


  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_freeze_provenance_v1.json


## 16. Freeze summary

In [23]:
print("=" * 78)
print("BUILD 1 — MCI SURVIVAL COHORT FREEZE (v1)")
print("=" * 78)
print(f"Assertions : {sum(a['passed'] for a in ASSERTIONS)} passed / "
      f"{sum(not a['passed'] for a in ASSERTIONS)} failed  (of {len(ASSERTIONS)})")
print(f"Reconciled : {'YES' if not DISCREPANT else 'NO — DISCREPANCY REPORT REQUIRED'}")
print()
print(recon.to_string(index=False))
print()
print(f"Flagged for Build 2 adjudication (dementia on/before anchor): {n_pre_dem}")
print(f"p-tau181: NOT integrated in v1 (see report).")
print()
print("Frozen outputs:")
for n, p in paths.items():
    print(f"  {n:46s} {OUTPUTS[n].shape[0]:>4d} x {OUTPUTS[n].shape[1]:<3d}  sha256={sha256(p)[:16]}...")
print(f"  {MANIFEST_PATH.name:46s} (provenance manifest)")
print()
print("AUTHORITATIVE FILE FOR MODELING (Person 2):")
print(f"  outputs/01c_mci_survival_cohort_freeze/mci_survival_primary_cohort_{VERSION}.tsv")
print(f"  duration = time_to_event_or_censor_days | event = event_indicator")
print(f"  predictors = entry_age + APOE4_COUNT (+ ptau217)")
print("=" * 78)

BUILD 1 — MCI SURVIVAL COHORT FREEZE (v1)
Assertions : 65 passed / 0 failed  (of 65)
Reconciled : YES

                   stage   n  events  target_n  target_events  diff_n  diff_events
         A. Broad anchor 535     NaN       535            NaN       0          NaN
   B. Survival-follow-up 410    86.0       410           86.0       0          0.0
C. Primary complete-case 401    85.0       401           85.0       0          0.0

Flagged for Build 2 adjudication (dementia on/before anchor): 8
p-tau181: NOT integrated in v1 (see report).

Frozen outputs:
  mci_survival_anchor_cohort_v1.tsv               535 x 78   sha256=64e66814726c2263...
  mci_survival_followup_cohort_v1.tsv             410 x 78   sha256=8203a70f7cc8d030...
  mci_survival_primary_cohort_v1.tsv              401 x 78   sha256=a15e2cb8606f676b...
  mci_survival_exclusion_log_v1.tsv               535 x 25   sha256=a3eafebcdd3e392b...
  mci_survival_cohort_flow_v1.tsv                  13 x 5    sha256=39b88d1a9f6cc3e4..